<a href="https://colab.research.google.com/github/kshamasahu706-wq/Customer-Churn-Feature-Engineering/blob/main/Customer_Churn_Responsible_Data_Card_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Churn Prediction & Responsible Data Card

**Project goal:** Explore the supplied customer dataset, document its responsible-use limitations, and build an *illustrative* churn-classification workflow.

> **Important limitation:** This file contains only 12 customer records (5 churned and 7 retained). Any model result will be highly uncertain and must not be treated as reliable business evidence. Use a much larger, representative dataset before making operational decisions.

## Dataset columns
- `customer_id`: customer identifier (excluded from model features)
- `tenure_months`: customer tenure in months
- `support_tickets`: number of support tickets
- `monthly_spend_inr`: monthly spend in INR
- `last_login_days`: days since last login
- `plan_type`: subscription plan
- `churned`: target (1 = churned, 0 = retained)

## How to use in Google Colab
1. Open this notebook in Google Colab.
2. Run the cells from top to bottom.
3. When prompted, upload `customer-churn-training(1).csv`.
4. Review the results and limitations before submitting.

## 1. Import libraries and upload the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

uploaded = files.upload()  # Select customer-churn-training(1).csv
csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)

if csv_name is None:
    raise ValueError("Please upload the customer churn CSV file.")

df = pd.read_csv(csv_name)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 2. Understand the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nSummary statistics:")
display(df.describe(include="all").T)

## 3. Data quality checks

In [ ]:
required_columns = [
    "customer_id", "tenure_months", "support_tickets",
    "monthly_spend_inr", "last_login_days", "plan_type", "churned"
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("Missing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

print("\\nTarget values:")
display(df["churned"].value_counts(dropna=False).to_frame("count"))

if not set(df["churned"].dropna().unique()).issubset({0, 1}):
    raise ValueError("The churned target must contain only 0 and 1.")

print("\\nTarget proportions:")
display(df["churned"].value_counts(normalize=True).rename("proportion").to_frame())

## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="churned")
plt.title("Customer churn class counts")
plt.xlabel("Churned (1 = Yes, 0 = No)")
plt.ylabel("Number of customers")
plt.show()

numeric_cols = ["tenure_months", "support_tickets", "monthly_spend_inr", "last_login_days"]
df[numeric_cols].hist(figsize=(10, 7), bins=8)
plt.suptitle("Distributions of numerical features")
plt.tight_layout()
plt.show()

print("Churn rate by plan type (descriptive only):")
display(pd.crosstab(df["plan_type"], df["churned"], normalize="index").round(3))

## 5. Outlier screening

The IQR rule below flags values for review; it does not automatically delete or change them. With only 12 rows, outlier conclusions are especially uncertain.

In [ ]:
outlier_rows = []
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flagged = df.index[(df[col] < lower) | (df[col] > upper)].tolist()
    outlier_rows.append({
        "feature": col, "lower_IQR_bound": lower,
        "upper_IQR_bound": upper, "flagged_row_indices": flagged
    })

display(pd.DataFrame(outlier_rows))

## 6. Responsible Data Card

This section follows the supplied Responsible Data Card template. Statements not supported by the CSV are explicitly marked as unknown.

### Dataset purpose
- **May support:** Exploratory analysis and educational experiments on predicting customer churn.
- **Must not decide:** Hiring, lending, insurance, eligibility, or other high-impact decisions. Do not automatically penalize or restrict customers based on a prediction.

### Provenance and permission
- The CSV contains 12 customer subscription/usage records.
- The file itself does not explain who collected the data, how it was collected, who is authorized to use it, or whether consent/licensing restrictions apply. These details must be confirmed with the data provider.

### Population and representation
- The data represents customers on Basic, Standard, and Pro plans.
- Only 12 records are present, so the sample may not represent the full customer population.
- Demographic groups such as age, gender, and location are not provided; representation and demographic fairness cannot be assessed.
- Target distribution: 7 retained (0) and 5 churned (1).

### Features and target
- Features: `tenure_months`, `support_tickets`, `monthly_spend_inr`, `last_login_days`, and `plan_type`.
- Target: `churned` (1 = churned, 0 = retained).
- `customer_id` is an identifier and is excluded from model training.
- Possible proxy concern: plan type and spending may be associated with customer circumstances. The CSV does not provide enough information to assess this.
- The dataset does not describe when features were recorded relative to churn, so temporal/label leakage cannot be ruled out.

### Quality checks
- The notebook calculates missing values, duplicate rows, duplicate IDs, class balance, and IQR outlier flags.
- Train/test separation is not a dependable evaluation strategy for such a tiny dataset; the notebook uses leave-one-out cross-validation for an exploratory demonstration only.

### Risks and safeguards
- **Overfitting / unstable estimates:** collect substantially more data and validate on a separate, later time period.
- **Privacy:** do not expose customer IDs; restrict access and follow applicable privacy rules.
- **False positives:** a retained customer may be incorrectly flagged as likely to churn; use human review and supportive, non-punitive outreach.
- **False negatives:** a customer who will churn may be missed; monitor errors and do not rely on the model alone.
- **Bias / representation:** obtain appropriate, lawfully collected data to assess group-level performance before deployment.
- **Misuse:** restrict use to customer-retention research and document authorized users.

### Intended evaluation
- Compare with a majority-class dummy baseline.
- Report accuracy, precision, recall, F1-score, confusion matrix, and error examples.
- Calibration and fairness evaluation are not reliable with this sample and missing demographic information; revisit them with an adequately sized dataset.

## 7. Prepare features for modelling

In [ ]:
# Remove the identifier and separate target from predictors.
X = df.drop(columns=["customer_id", "churned"])
y = df["churned"].astype(int)

numeric_features = ["tenure_months", "support_tickets", "monthly_spend_inr", "last_login_days"]
categorical_features = ["plan_type"]

print("Feature matrix shape:", X.shape)
print("Target distribution:")
display(y.value_counts())

## 8. Baseline and illustrative model evaluation

Leave-One-Out Cross-Validation (LOOCV) trains on all but one row and predicts the held-out row. This avoids choosing a tiny random test set, but it does **not** solve the fundamental problem of having only 12 records. Results are for learning only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

baseline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

logistic_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

cv = LeaveOneOut()

baseline_pred = cross_val_predict(baseline, X, y, cv=cv)
model_pred = cross_val_predict(logistic_model, X, y, cv=cv)

def show_metrics(name, y_true, y_pred):
    print(f"\\n{name}")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_true, y_pred, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_true, y_pred, zero_division=0), 3))
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print(classification_report(y_true, y_pred, zero_division=0))

show_metrics("Majority-class baseline (LOOCV)", y, baseline_pred)
show_metrics("Logistic Regression (LOOCV)", y, model_pred)

## 9. Review prediction errors

The table below shows out-of-fold predictions: each row was predicted by a model that did not train on that row.

In [ ]:
error_review = df[["customer_id", "churned"]].copy()
error_review["predicted_churned"] = model_pred
error_review["correct"] = error_review["churned"] == error_review["predicted_churned"]
display(error_review)

print("Incorrectly predicted records:")
display(error_review.loc[~error_review["correct"]])

## 10. Final conclusion

This small dataset is useful for practising data inspection, responsible data documentation, preprocessing, and classification evaluation. It is **not sufficient to build or validate a dependable churn prediction system**.

Before real-world use:
1. Obtain permission and document data provenance.
2. Collect a much larger and more representative dataset.
3. Confirm that all predictors are available before the churn outcome.
4. Evaluate on a time-based holdout dataset.
5. Review false positives and false negatives with business stakeholders.
6. Assess privacy, fairness, calibration, and ongoing model performance.
7. Keep a human in the decision loop.

**End of project.**